In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *
from models.lstm_xgb import LSTM_XGB

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_NAME = "LSTM_XGB"


device: NVIDIA A30


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }

In [3]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, device, epochs=70, lr=1e-2, seed=0)
    results.append(r)



=== Scenario 1 (LSTM_XGB) ===
  LSTM params: 4,744  breakdown: {'lstm': 4480, 'head': 264}
  [stage1] ep   1 | loss 1.8172 acc 0.248 | val loss 1.9040 acc 0.194
  [stage1] ep  10 | loss 1.5261 acc 0.366 | val loss 1.5391 acc 0.355
  [stage1] ep  20 | loss 0.4013 acc 0.810 | val loss 0.3815 acc 0.826
  [stage1] ep  30 | loss 0.3471 acc 0.832 | val loss 0.3778 acc 0.833
  [stage1] ep  40 | loss 0.3189 acc 0.854 | val loss 0.3284 acc 0.849
  [stage1] ep  50 | loss 0.2628 acc 0.889 | val loss 0.2799 acc 0.885
  [stage1] ep  60 | loss 0.2346 acc 0.903 | val loss 0.2684 acc 0.897
  [stage1] ep  70 | loss 0.2029 acc 0.920 | val loss 0.2477 acc 0.914
  TEST: acc=0.9131  P=0.9132  R=0.9131  F1=0.9130
  COMPUTE: stage1(LSTM)=56.1s  stage2(XGB)=28.5s  total=84.6s  inf=0.044ms/sample

=== Scenario 2 (LSTM_XGB) ===
  LSTM params: 4,744  breakdown: {'lstm': 4480, 'head': 264}
  [stage1] ep   1 | loss 1.9076 acc 0.219 | val loss 1.7163 acc 0.314
  [stage1] ep  10 | loss 0.9491 acc 0.590 | val loss 0

In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)


 scenario    model  n_train  accuracy  precision  recall     f1  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 LSTM_XGB     7858    0.9131     0.9132  0.9131 0.9130      4744      84.59             0.0445        311.4
        2 LSTM_XGB     3939    0.8612     0.8606  0.8612 0.8607      4744      50.04             0.0352        168.2
        3 LSTM_XGB      786    0.8162     0.8164  0.8162 0.8153      4744      24.63             0.0482         82.8
        4 LSTM_XGB      391    0.3919     0.3968  0.3919 0.3935      4744      14.06             0.0353         82.8
        5 LSTM_XGB      238    0.3988     0.3982  0.3988 0.3965      4744      10.60             0.0382         82.8
        6 LSTM_XGB       77    0.3731     0.3676  0.3731 0.3664      4744      12.33             0.0027         82.8


In [5]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (LSTM_XGB, n_train=7858, acc=0.913)
[[174   1   0  21   0   0   3   1]
 [  0 190   0   2   0   0   0   8]
 [  0   0 199   0   0   1   0   0]
 [ 16   1   0 183   0   0   0   0]
 [  0   0   0   0 200   0   0   0]
 [  0   1   3   0   0 160   0  36]
 [  1   1   0   2   0   0 196   0]
 [  0   8   0   0   0  33   0 159]]

Scenario 2  (LSTM_XGB, n_train=3939, acc=0.861)
[[145   2   0  50   2   0   0   1]
 [  1 188   0   0   0   1   0  10]
 [  0   0 196   0   0   4   0   0]
 [ 58   0   0 136   1   0   5   0]
 [  0   0   0   0 200   0   0   0]
 [  0   0   7   0   0 158   0  35]
 [  0   0   0   5   2   0 193   0]
 [  0   9   1   0   0  28   0 162]]

Scenario 3  (LSTM_XGB, n_train=786, acc=0.816)
[[144   2   0  52   0   1   0   1]
 [  3 181   0   1   0   3   0  12]
 [  0   0 189   1   0   6   0   4]
 [ 44   2   0 143   1   0  10   0]
 [  0   0   1   0 197   2   0   0]
 [  0   3  10   0   4 123   0  60]
 [  0   0   1  12   0   0 187   0]
 [  1  21   4   0   1  31   0 142]]

Scenario 4